[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tnecniv-Teikram/hyperspectral-course-2026/blob/main/exercises/ex_02_land_use_land_cover.ipynb)

*Running in Colab? Run the setup cell below first — it clones the course and installs what this notebook needs. Running locally? It does nothing.*

# 🏙️ Exercises: Land use & land cover change
### Space Academy 2026 · Hyperspectral course · Week 2 — Project scoping and use case definition (Practical)

Put the index-threshold land-cover map of notebook 02 to the test: compare it with an independent reference, tune it, replace it with unsupervised clustering on the full spectrum, and look inside the 'built-up' class at the different roof materials of Ho Chi Minh City.

These exercises accompany the theme notebook [`02_land_use_land_cover_change.ipynb`](../../notebooks/02_land_use_land_cover_change.ipynb). Run that notebook first
(or at least read it) — here we reuse its scene and indices, loaded with the course helper module `hsi_utils`.

> Work through the exercises in order — they get harder (⭐ → ⭐⭐⭐). Each has a hint; worked solutions are in `exercises/solutions/`.

**What you will practise**
- Area statistics from a classification map
- Comparing a map with a reference (confusion matrix, agreement, recall)
- Choosing thresholds from data — and why they don't transfer
- Unsupervised full-spectrum clustering versus index thresholds
- Urban material spectra and atmospheric residuals

**Data**
- `20250407_035509_25_4001` — southern Ho Chi Minh City, Vietnam (7 April 2025)
- `data/labels/…_worldcover2021_30m.tif` — ESA WorldCover 2021, aggregated to the Tanager 30 m grid

In [ ]:
# --- Setup: works both in Google Colab and in a local environment --------------------
# In Colab this clones the course repository, installs the geospatial packages that Colab
# does not ship, and moves into the course folder so the notebook finds hsi_utils and data/.
# Locally it does nothing at all — your own environment is already active.
import os
import pathlib
import subprocess
import sys

REPO = "Tnecniv-Teikram/hyperspectral-course-2026"          # change with: python tools/set_repo_url.py <owner>/<repo>

if "google.colab" in sys.modules:
    course = pathlib.Path("/content/hyperspectral-course")
    if not course.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q",
                        f"https://github.com/{REPO}.git", str(course)], check=True)
    os.chdir(course)
    sys.path.insert(0, str(course))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "h5py", "rasterio", "rioxarray", "xarray", "geopandas"], check=True)
    print("Colab ready — course at", course)
else:
    print("Local environment:", sys.executable.split("/")[-3])


## Setup

In [ ]:
# hsi_utils.py lives in the course folder (.. from here)
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# On Google Colab: upload hsi_utils.py next to this notebook, change the line above to
# sys.path.insert(0, ".") and uncomment:
# %pip install -q h5py numpy scipy matplotlib scikit-learn pandas rasterio requests

import numpy as np
import matplotlib.pyplot as plt
import hsi_utils as hu

plt.rcParams["figure.dpi"] = 90

Load the scene and the bands used in the theme notebook. `hu.get_scene()` finds the local HDF5 or downloads it (~1 GB) from Planet's STAC catalogue.

In [ ]:
import pandas as pd

SCENE_ID = "20250407_035509_25_4001"   # Ho Chi Minh City, Vietnam (urban), 7 April 2025
scene = hu.get_scene(SCENE_ID)
print(scene.summary())

valid = scene.valid_mask()
R_green, R_red, R_nir, R_swir = (scene.read_band(w) for w in (560, 665, 830, 1610))   # as in notebook 02
NDVI = hu.nd(R_nir, R_red)
NDBI = hu.nd(R_swir, R_nir)
MNDWI = hu.nd(R_green, R_swir)
PIXEL_HA = 0.09

# The threshold classification of notebook 02 (same rules, same order)
CLASSES = ["Water", "Vegetation", "Built-up", "Bare soil"]
lc = np.full(NDVI.shape, 255, dtype=np.uint8)
lc[MNDWI > 0.1] = 0
lc[NDVI > 0.2] = 1
lc[NDBI > 0.0] = 2
lc[(NDBI <= 0) & (NDVI <= 0.2) & (MNDWI <= 0.1)] = 3
lc[~valid] = 255

# ESA WorldCover 2021 on the Tanager grid: majority class + purity (how many of the nine 10 m cells agree)
wc, purity = hu.load_labels(SCENE_ID)

from matplotlib.colors import ListedColormap
CMAP4 = ListedColormap(["#2196F3", "#4CAF50", "#F44336", "#F5CBA7"])
wc_cmap, wc_norm, wc_handles = hu.label_cmap([10, 30, 40, 50, 60, 80])
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
axes[0].imshow(scene.rgb()); axes[0].set_title("True colour")
axes[1].imshow(np.where(lc == 255, np.nan, lc), cmap=CMAP4, vmin=0, vmax=3, interpolation="nearest")
axes[1].set_title("Threshold classification (notebook 02)")
axes[2].imshow(np.where(valid & (wc > 0), wc, np.nan), cmap=wc_cmap, norm=wc_norm, interpolation="nearest")
axes[2].set_title("ESA WorldCover 2021 (30 m majority)")
axes[2].legend(handles=wc_handles, loc="lower left", fontsize=7)
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

## Exercise 1 — Land-cover statistics in km²  ⭐

Using the threshold map `lc` (0 = water, 1 = vegetation, 2 = built-up, 3 = bare soil, 255 = no data):

1. Make a small table (pandas `DataFrame`) with the pixel count, area in km² and % of the valid area per class.
2. Ho Chi Minh City is one of the most densely built cities in Southeast Asia — does the built-up share
   surprise you? Look at the true-colour image: which part of the scene is the dense city?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

`classes, counts = np.unique(lc[valid], return_counts=True)`; one pixel = 0.09 ha = 0.0009 km².

</details>

## Exercise 2 — How well do the thresholds agree with WorldCover?  ⭐⭐

Compare the threshold map with ESA WorldCover (2021) on **pure** pixels only (`purity == 9`, i.e. all nine
10 m WorldCover cells inside the 30 m pixel have the same class).

1. Translate the WorldCover classes to the four classes: 80 → Water; 10, 20, 30, 40, 90, 95 → Vegetation;
   50 → Built-up; 60 → Bare soil.
2. Build a confusion matrix (rows = WorldCover, columns = thresholds) and compute the overall agreement and
   the per-class *recall* (the fraction of each WorldCover class that the thresholds got right).
3. Which confusion dominates? Look at where it happens on the map.

We say **agreement**, not accuracy: WorldCover is a 2021 product with its own errors, and the city has changed.

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

Build the translated reference with a loop over a dictionary `{wc_code: new_class}`. `pd.crosstab(ref[ok], lc[ok])`
gives the confusion matrix; recall = diagonal / row sum.

</details>

## Exercise 3 — Choosing the NDBI threshold from data  ⭐⭐

The value `NDBI > 0` is a textbook default. Find the threshold that best reproduces WorldCover **built-up**
(class 50) on pure pixels:

1. For thresholds from −0.25 to 0.15 (step 0.025), compute precision, recall and F1 of `NDBI > t` against
   WorldCover built-up.
2. Plot the three curves and mark the best-F1 threshold.
3. Is the "best" threshold transferable to another city? Why (not)?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

With `truth = wc == 50` and `pred = NDBI > t` on the mask `ok = valid & (purity == 9) & (wc > 0)`:
TP = `(pred & truth).sum()`, precision = TP / pred.sum(), recall = TP / truth.sum(), F1 = 2PR / (P + R).

</details>

## Exercise 4 — Let the spectra speak: k-means instead of thresholds  ⭐⭐⭐

Replace the hand-made thresholds by unsupervised clustering on the **full spectrum**:

1. Read the whole scene with `stride=2` (every 2nd pixel, to save memory) and reduce it to 15 PCA components.
2. Run k-means with k = 10 (more clusters than classes: several spectral clusters can map to one class).
3. Give each cluster the four-class label that is most common among its pure WorldCover pixels
   (majority vote) — this uses the reference only to *name* clusters, not to train them.
4. Compute the agreement with WorldCover on pure pixels, as in Exercise 2, and compare it with the thresholds
   on the same stride-2 grid.

Is it a fair comparison? What information did the clustering use that the thresholds did not?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

`cube, wl = scene.read_cube(stride=2)`, `X, okpix = hu.cube_to_pixels(cube)`, fit PCA and k-means on a random
subset of ~30,000 pixels (faster), then `predict` all pixels. Subsample the reference with `[::2, ::2]`.

</details>

## Exercise 5 — Roof and pavement materials  ⭐⭐

"Built-up" is not one material. Within WorldCover built-up pixels (class 50, NDVI < 0.15) select three groups:

- **bright roofs**: the brightest 1% (mean reflectance of 480, 550, 665, 830 and 1610 nm);
- **red / orange surfaces**: the top 0.5% of the ratio R₆₄₀ / R₅₅₀ among pixels brighter than the median;
- **dark surfaces**: the darkest 5%.

Plot the mean spectrum of each group plus a tree-cover reference (WorldCover 10, NDVI > 0.7). Which spectral
features could tell the materials apart, and which dips are *not* material features at all?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

Build a mask per group, then `rows, cols = np.nonzero(mask)`, read the bounding window with
`scene.read_cube((rows.min(), rows.max() + 1, cols.min(), cols.max() + 1))` and index it with
`cube[:, rows - rows.min(), cols - cols.min()]`.

</details>

## Exercise 6 — Reflection — what does '80% agreement' mean?  ⭐

Answer in a short markdown cell:

1. List three reasons why the threshold map and WorldCover can disagree even where the threshold map is right.
2. What share of the valid pixels are *pure* (purity = 9)? Why did we evaluate only on pure pixels — and what
   bias does that introduce into the agreement score?

In [ ]:
# compute the share of pure pixels here, then answer in a markdown cell

<details><summary>💡 Hint</summary>

`(purity[valid] == 9).mean()`. Think about time (2021 vs 2025), resolution (10 m vs 30 m), class definitions
("tree cover" in a garden city) and WorldCover's own accuracy (~75% globally).

</details>

## Key takeaways

- Always compare a map with an independent reference — and call it agreement unless the reference is ground truth.
- Index thresholds are scene-specific models; tune them on one part of the data and test on another.
- Full-spectrum clustering captures materials that three indices cannot.
- 'Built-up' contains distinct materials (metal, tiles, asphalt) with diagnostic features such as iron-oxide absorption.
- Evaluating only on pure pixels gives optimistic scores.

---

> 📖 Data: Tanager — CC-BY-4.0 © Planet Labs PBC  
> 📖 Labels: ESA WorldCover 2021 v200 — CC-BY 4.0